In [ ]:
import json
import jieba
import re

voc = {}, {}
X, Y = zip(
    *[[torch.tensor([voc.setdefault(i, len(voc)) for i in jieba.cut(re.sub(r"\s", "", i.lower()))]) for i, voc in zip(i.values(), voc)] for i in json.load(open("../chatgpt/qa/backend/qa.json"))]
)

In [ ]:
m = 64
m = Seq(
    Zip(
        Stack(
            Embedding(len(voc[0]), m),
            lambda x: functional.interpolate(x.T[None], 32, mode="linear")[0].T,
        ),
        Seq(
            Stack(
                RandHot(len(voc[1])),
                F(torch.matmul, Embedding(len(voc[1]), m).weight),
                lambda x: functional.interpolate(x.T[None], 32, mode="linear")[0].T,
            ),
        ),
        Identity(),
    ),
    lambda x: [x[0] + x[1], x[2]],
    Zip(
        Seq(
            *(
                Add(
                    Seq(
                        Nn(m, m, m, pos=TriPos, act=SiLU(True)),
                        AttentionMin,
                    ),
                    Linear(m, m, False),
                )
                for i in range(3)
            ),
            Nn(m, m, m, pos=TriPos, act=SiLU(True)),
        ),
        Identity(),
    ),
    lambda x: zip(*x),
    Map(
        F(lambda x: functional.interpolate(x[0].T[None], x[1], mode="linear")[0].T),
        Nn(0, m, len(voc[1]), act=SiLU(True)),
    ),
)
o = torch.optim.Adam(m.parameters())
O = torch.optim.lr_scheduler.OneCycleLR(o, 0.001, 100000, pct_start=0.01, div_factor=9, final_div_factor=999)
log = Log()
los = []
acc = []

In [ ]:
while O.last_epoch < O.total_steps:
    i = torch.randint(len(X), [16])
    x = [X[i] for i in i]
    t = [Y[i] for i in i]
    l = [len(i) for i in t]
    y = m([x, t, l])
    l = sum(functional.cross_entropy(y, t) * len(t) for y, t in zip(y, t))
    l.backward()
    o.step()
    o.zero_grad()
    O.step()
    if O.last_epoch % 100 >= 98:
        los.append(l)
        acc.append(sum((y.argmax(-1) == t).sum() for y, t in zip(y, t)) / sum(map(len, t)))
        if not (O.last_epoch + 1) % 100:
            with torch.no_grad():
                log(los=sum(los) / len(los), acc=sum(acc) / len(acc))
            los.clear()
            acc.clear()IPython

In [ ]:
import pandas as pd


def f(x, voc):
    return ["".join(tuple(voc)[i] for i in x) for x in x]


i = torch.randint(len(X), [32])
x = [X[i] for i in i]
t = [Y[i] for i in i]
y = [torch.zeros_like(i) for i in t]
l = [len(i) for i in t]
for p in torch.linspace(1, 0, 1000) ** 5e-4:
    y = [RandHot(i, len(voc[1]), lambda _: p[None]) for i in y]
    with torch.no_grad():
        y = [i.softmax(-1) for i in m([x, y, l])]
pd.set_option("display.max_colwidth", None)
pd.DataFrame({"x": f(x, voc[0]), "t": f(t, voc[1]), "y": f(map(F(torch.argmax, 1), y), voc[1])})

In [ ]:
t = torch.rand(1024, 100)
p = torch.tensor([0.99]) ** .2
t *= p[:, None] / t.sum(1, True)
t[:, 0] += 1 - p
1 - t.argmax(1).bool().sum() / len(t)